# webgpu-dna on Kaggle's free GPU (WGSL via wgpu-py)

Kaggle gives a free CUDA GPU (T4 x2 / P100, ~30 GPU-hrs/week). Our physics is
**WGSL** (WebGPU), not CUDA — so the bridge is [`wgpu-py`](https://github.com/pygfx/wgpu-py),
Python bindings to the same `wgpu-native` (Rust) runtime the roadmap's
`webgpu-dna-native` would use. On Linux it reaches the GPU through **Vulkan**.

**This notebook is a PROBE**, not the full simulation. It answers one question:
*does `wgpu-py` actually acquire Kaggle's GPU and run a WGSL compute shader?*

## ⚠️ The catch with datacenter GPUs
Tesla cards on Kaggle/Colab often run **compute-only NVIDIA drivers** that ship
CUDA but **no Vulkan** component. Since every WebGPU implementation uses
Vulkan/Metal/D3D (never CUDA), `wgpu-py` then falls back to Mesa's CPU software
rasterizer (`llvmpipe`). Cell 1 detects this and, if the NVIDIA Vulkan driver
*is* present, registers it so the loader can find the real GPU.

## Before running
1. Right sidebar → **Settings → Accelerator → GPU T4 x2** (or P100).
2. **Run All.** If you change the accelerator, do **Run → Restart & Run All**
   so the Vulkan env in Cell 1 is set before `wgpu` first imports.

In [ ]:
# === Cell 1: env + NVIDIA Vulkan ICD detection (MUST run before importing wgpu) ===
import os, subprocess, json, glob
!pip -q install wgpu numpy
!apt-get -qq install -y libvulkan1 vulkan-tools >/dev/null 2>&1 || true

os.environ['XDG_RUNTIME_DIR'] = '/tmp'   # silences the surfaceless-platform warning

def sh(c):
    return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()

print('nvidia-smi:', sh('nvidia-smi -L') or 'no GPU visible (CUDA)')

# Is the NVIDIA *Vulkan* driver library present? (compute-only drivers omit it)
cand = sh("ldconfig -p | grep -oE '/[^ ]*libGLX_nvidia\\.so[^ ]*'").split()
cand += glob.glob('/usr/lib/x86_64-linux-gnu/libGLX_nvidia.so*')
cand += glob.glob('/usr/lib/**/libGLX_nvidia.so*', recursive=True)
cand = sorted(set(c for c in cand if c))
print('NVIDIA Vulkan lib:', cand or 'NONE FOUND')

if cand:
    icd = {'file_format_version': '1.0.0',
           'ICD': {'library_path': cand[0], 'api_version': '1.3.0'}}
    json.dump(icd, open('/tmp/nvidia_icd.json', 'w'))
    os.environ['VK_ICD_FILENAMES'] = '/tmp/nvidia_icd.json'  # older loader
    os.environ['VK_DRIVER_FILES']  = '/tmp/nvidia_icd.json'  # newer loader
    print('-> Registered NVIDIA Vulkan ICD:', cand[0])
    print('   If the probe still shows llvmpipe, the GPU does not expose Vulkan compute.')
else:
    print('-> Compute-only CUDA driver: no NVIDIA Vulkan component on this image.')
    print('   wgpu/WebGPU (Vulkan-only on Linux) cannot use this GPU. See the fallback in README.')

In [ ]:
# === Cell 2: which adapter does wgpu-py get? ===
import wgpu, numpy as np
print('wgpu-py version:', wgpu.__version__)
try:
    adapter = wgpu.gpu.request_adapter_sync(power_preference='high-performance')
    info = dict(adapter.info)
    print('Adapter info:')
    for k, v in info.items():
        print(f'  {k:16} {v}')
    atype = str(info.get('adapter_type', '')).lower()
    dev = str(info.get('device', '') or info.get('description', '')).lower()
    on_gpu = ('cpu' not in atype) and ('llvmpipe' not in dev) and ('software' not in dev)
    print()
    if on_gpu:
        print('GPU ACQUIRED:', info.get('device') or info.get('description'))
        print('   -> wgpu-py runs WGSL on Kaggle\u2019s real GPU. The host port is worth doing.')
    else:
        print('STILL ON SOFTWARE (llvmpipe / CPU). This GPU is not Vulkan-accessible.')
        print('   -> Free datacenter GPUs (Kaggle/Colab) are CUDA-only; the free-WebGPU path')
        print('      is the WebRTC swarm (volunteer consumer browsers). See README.')
except Exception as e:
    print('No adapter:', e)
    print('   (VK_ICD_FILENAMES forced NVIDIA-only and it provided no device -> compute-only driver.)')

In [ ]:
# === Cell 3: run an actual WGSL compute shader and verify (works on any adapter) ===
from wgpu.utils.compute import compute_with_buffers

WGSL = '''
@group(0) @binding(0) var<storage, read_write> data: array<f32>;
@compute @workgroup_size(64)
fn main(@builtin(global_invocation_id) gid: vec3<u32>) {
  let i = gid.x;
  if (i < arrayLength(&data)) { data[i] = data[i] * 2.0 + 1.0; }
}
'''

n = 4096
data = np.arange(n, dtype=np.float32)
out = compute_with_buffers(input_arrays={0: data},
                           output_arrays={0: (n, 'f')},
                           shader=WGSL, n=(n // 64, 1, 1))
result = np.frombuffer(out[0], dtype=np.float32)
expected = data * 2.0 + 1.0
ok = np.allclose(result, expected)
print('in  :', data[:5])
print('out :', result[:5])
print('want:', expected[:5])
print('WGSL executed correctly' if ok else 'MISMATCH — WGSL did not run as expected')

## Reading the result
- **Cell 2 shows the Tesla GPU** + Cell 3 passes → `wgpu-py` runs our WGSL on
  Kaggle's free GPU. Next step: port the TS host orchestration
  (`src/gpu/{buffers,pipelines,dispatch}.ts`) to Python; the shaders below
  compile unchanged.
- **Cell 2 stays on `llvmpipe`/CPU** even after Cell 1 → this is a compute-only
  CUDA driver with no Vulkan. Free datacenter GPUs can't run WebGPU; the free
  path is the **WebRTC swarm** (volunteer consumer browsers). Cell 3 still
  passes — it just ran on the CPU software adapter.

In [ ]:
# === Cell 4: pull the actual shaders (verbatim-portable WGSL) ===
!git clone --depth 1 -q https://github.com/abgnydn/webgpu-dna /kaggle/working/webgpu-dna 2>/dev/null || (cd /kaggle/working/webgpu-dna && git pull -q)
import os
root = '/kaggle/working/webgpu-dna'
for f in ['src/shaders/primary.wgsl', 'src/shaders/secondary.wgsl', 'src/shaders/helpers.wgsl', 'public/cross_sections.wgsl']:
    p = os.path.join(root, f)
    if os.path.exists(p):
        print(f'{f:34} {os.path.getsize(p)//1024:5} KB')
print('\nThese compile under wgpu-native unchanged; only the TS host orchestration needs a Python port.')